In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krupalpatel07/palantir-technologies-inc/PLTR.csv


In [2]:
# ==========================================================
# 1. IMPORT LIBRARIES
# ==========================================================


import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestRegressor

from IPython.display import display, HTML

pio.renderers.default = "iframe"

In [3]:
# ==========================================================
# 2. LOAD DATA
# ==========================================================

file_path = "/kaggle/input/datasets/krupalpatel07/palantir-technologies-inc/PLTR.csv"

df = pd.read_csv(file_path)

df.columns = [c.lower() for c in df.columns]

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.set_index("date", inplace=True)

In [4]:
# ==========================================================
# 3. HEADER
# ==========================================================

def fusion_header(title):

    display(HTML(f"""
    <div style="
        background:
        linear-gradient(135deg,
        #050505,
        #112240,
        #1b3c73,
        #2dd4ff);
        padding:30px;
        border-radius:22px;
        margin-top:18px;
        margin-bottom:18px;
        box-shadow:0px 0px 45px rgba(45,212,255,.35);
    ">
    <h1 style="
        color:white;
        text-align:center;
        font-size:40px;
        font-family:Trebuchet MS;
        letter-spacing:4px;">
        {title}
    </h1>
    </div>
    """))

fusion_header("🛰 PALANTIR Intelligence Fusion Operations Center")


In [5]:
# ==========================================================
# 4. COMMAND BRIEFING
# ==========================================================

fusion_header("🎯 Mission Command Briefing")

df["returns"] = df["close"].pct_change()

mission_return = (
(
df["close"].iloc[-1]
/
df["close"].iloc[0]
)-1
)*100

annual_vol = (
df["returns"].std()
*np.sqrt(252)
*100
)

intel_efficiency = (
df["returns"].mean()
/
df["returns"].std()
)*np.sqrt(252)

dashboard = pd.DataFrame({

"Metric":[
"Mission Return %",
"Risk Level %",
"Intel Efficiency"
],

"Value":[
round(mission_return,2),
round(annual_vol,2),
round(intel_efficiency,2)
]

})

fig = px.sunburst(
dashboard,
path=["Metric"],
values="Value",
color="Value",
title="Mission Dashboard"
)

fig.show()


In [6]:
# ==========================================================
# 5. SIGNAL INTERCEPT ENGINE
# ==========================================================

fusion_header("📡 Signal Intercept Engine")

df["intel7"] = df["close"].pct_change(7)

df["intel30"] = df["close"].pct_change(30)

df["intel90"] = df["close"].pct_change(90)

df["signal_index"] = (

df["intel7"]*0.45+

df["intel30"]*0.35+

df["intel90"]*0.20

)

fig = px.line(
df,
y="signal_index",
title="Intercepted Signal Index"
)

fig.show()


In [7]:
# ==========================================================
# 6. THREAT HEATMAP
# ==========================================================

fusion_header("🔥 Threat Activity Scanner")

df["threat_score"] = (

np.log1p(df["volume"])

*

abs(df["returns"])

*

100

)

fig = px.density_heatmap(

df.reset_index(),

x="date",

y="threat_score",

title="Threat Density"

)

fig.show()


In [8]:
# ==========================================================
# 7. SURVEILLANCE RADAR
# ==========================================================

fusion_header("🛰 Surveillance Radar")

df["ema20"]=df["close"].ewm(span=20).mean()

df["ema80"]=df["close"].ewm(span=80).mean()

df["radar_gap"]=df["ema20"]-df["ema80"]

fig=go.Figure()

fig.add_trace(

go.Scatter(

x=df.index,

y=df["radar_gap"],

fill="tozeroy",

name="Radar Gap"

)

)

fig.update_layout(
title="Surveillance Coverage"
)

fig.show()


In [9]:
# ==========================================================
# 8. CONFIDENCE MATRIX
# ==========================================================

fusion_header("🧠 AI Confidence Matrix")

trend=(
df["close"]
/
df["close"].rolling(120).mean()
)

volume=(
df["volume"]
.rank(pct=True)
)

stability=(
1-
df["returns"]
.rolling(20)
.std()
.rank(pct=True)
)

df["confidence"]=(

trend.rank(pct=True)*0.40+

volume*0.30+

stability*0.30

)

fig=px.area(
df,
y="confidence",
title="Confidence Matrix"
)

fig.show()


In [10]:
# ==========================================================
# 9. OPERATIONAL STATUS
# ==========================================================

fusion_header("⚔ Operational Status Engine")

momentum=df["close"].pct_change(25)

vol=df["returns"].rolling(20).std()

conditions=[

(momentum>0.15),

(momentum>0),

(momentum<0)&(vol<vol.median()),

(momentum<0)&(vol>vol.median())

]

choices=[

"Strategic Advance",

"Recon Mode",

"Containment",

"High Alert"

]

df["operation"]=np.select(
conditions,
choices,
default="Monitoring"
)

fig=px.scatter(
df,
x=df.index,
y="close",
color="operation",
title="Operational Status"
)

fig.show()


In [11]:
# ==========================================================
# 10. AI CLUSTER DISCOVERY
# ==========================================================

fusion_header("🤖 AI Pattern Discovery")

cluster=df[
[
"signal_index",
"threat_score",
"confidence"
]
].fillna(0)

scaled=QuantileTransformer(
output_distribution="normal",
random_state=42
).fit_transform(cluster)

model=DBSCAN(
eps=0.65,
min_samples=12
)

df["cluster"]=model.fit_predict(scaled)

fig=px.scatter(
df,
x=df.index,
y="close",
color=df["cluster"].astype(str),
title="Hidden Intelligence Clusters"
)

fig.show()


In [12]:
# ==========================================================
# 11. PREDICTIVE IMPORTANCE
# ==========================================================

fusion_header("🔍 Predictive Intelligence Engine")

temp=df.copy()

temp["future"]=temp["close"].shift(-1)

temp=temp.dropna()

X=temp[
[
"signal_index",
"threat_score",
"confidence"
]
]

y=temp["future"]

model=RandomForestRegressor(
n_estimators=200,
random_state=42
)

model.fit(X,y)

importance=pd.DataFrame({

"Factor":X.columns,

"Importance":model.feature_importances_

})

fig=px.bar(
importance,
x="Factor",
y="Importance",
title="AI Feature Importance"
)

fig.show()


In [13]:
# ==========================================================
# 12. TARGET ACQUISITION
# ==========================================================

fusion_header("🎯 Target Acquisition Engine")

signal=(

(df["confidence"]>
df["confidence"].rolling(40).mean())

&

(df["signal_index"]>0)

)

df["target"]=signal.astype(int)

fig=go.Figure()

fig.add_trace(

go.Scatter(

x=df.index,

y=df["close"],

name="Price"

)

)

fig.add_trace(

go.Scatter(

x=df.index[df["target"]==1],

y=df["close"][df["target"]==1],

mode="markers",

marker=dict(size=9),

name="Target Locked"

)

)

fig.update_layout(
title="Target Acquisition Zones"
)

fig.show()


In [14]:
# ==========================================================
# 13. FINAL INTELLIGENCE REPORT
# ==========================================================

fusion_header("📘 Intelligence Assessment")

print("""

1. Signal Intercept measures multi-timeframe momentum.

2. Threat Scanner detects abnormal market intensity.

3. Surveillance Radar tracks structural trend shifts.

4. Confidence Matrix evaluates market reliability.

5. Operational Status classifies strategic environments.

6. AI Discovery uncovers hidden behavioral clusters.

7. Random Forest identifies dominant predictive factors.

8. Target Acquisition highlights high-conviction opportunities.

9. Palantir market behavior resembles an adaptive intelligence network.

""")



1. Signal Intercept measures multi-timeframe momentum.

2. Threat Scanner detects abnormal market intensity.

3. Surveillance Radar tracks structural trend shifts.

4. Confidence Matrix evaluates market reliability.

5. Operational Status classifies strategic environments.

6. AI Discovery uncovers hidden behavioral clusters.

7. Random Forest identifies dominant predictive factors.

8. Target Acquisition highlights high-conviction opportunities.

9. Palantir market behavior resembles an adaptive intelligence network.


